# SuperKart — Cleaned & Refactored Notebook

This notebook has been refactored to: 
- Move and centralize imports at the top
- Remove empty/placeholder cells and redundant data-loading cells
- Add reusable functions for EDA and evaluation
- Centralize preprocessing into a single ColumnTransformer
- Train multiple models in a loop and save pipelines/models to disk
- Reduce verbosity (XGBoost) and make pip installs quiet


In [ ]:
# Install required packages (quiet)
# NOTE: Restart the kernel after changing package versions if needed
!pip install numpy==2.0.2 pandas==2.2.2 scikit-learn==1.6.1 matplotlib==3.10.0 seaborn==0.13.2 joblib==1.4.2 xgboost==2.1.4 requests==2.32.4 huggingface_hub==0.34.0 --quiet

In [ ]:
# Standard imports grouped at top
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import make_column_transformer, ColumnTransformer
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
import joblib

pd.set_option('display.max_columns', None)

# Create output directories for models and artifacts
os.makedirs('models', exist_ok=True)
os.makedirs('artifacts', exist_ok=True)


## Data loading (single canonical location)

In [ ]:
# Update the path if the CSV is located elsewhere
DATA_PATH = '/content/SuperKart.csv'
data = pd.read_csv(DATA_PATH)
data.head()

## Reusable EDA functions

In [ ]:
def quick_univariate_target(df, target='Product_Store_Sales_Total'):
    print('=== Target summary ===')
    print(df[target].describe())
    plt.figure(figsize=(10,5))
    sns.histplot(df[target], bins=50, kde=True)
    plt.title(f'Distribution of {target}')
    plt.show()

def quick_numeric_relations(df, target='Product_Store_Sales_Total'):
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    if target in numeric_cols:
        numeric_cols.remove(target)
    for col in numeric_cols:
        plt.figure(figsize=(6,4))
        sns.scatterplot(x=col, y=target, data=df, alpha=0.5)
        plt.title(f'{target} vs {col}')
        plt.show()

def quick_categorical_counts(df, exclude_ids=['Product_Id','Store_Id']):
    categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
    categorical_cols = [c for c in categorical_cols if c not in exclude_ids]
    for col in categorical_cols:
        plt.figure(figsize=(8,4))
        sns.countplot(y=col, data=df, order=df[col].value_counts().index)
        plt.title(f'Distribution of {col}')
        plt.show()

# Example usage (uncomment to run interactively)
# quick_univariate_target(data)
# quick_numeric_relations(data)
# quick_categorical_counts(data)


## Preprocessing (single ColumnTransformer + Pipeline)

In [ ]:
# Define features and target
TARGET = 'Product_Store_Sales_Total'
FEATURES = [c for c in data.columns if c != TARGET]

# Identify numerical and categorical columns
numerical_cols = data.select_dtypes(include=['number']).columns.tolist()
if TARGET in numerical_cols:
    numerical_cols.remove(TARGET)
categorical_cols = [c for c in data.select_dtypes(include=['object']).columns.tolist() if c not in ['Product_Id','Store_Id']]

# Build transformers
num_transformer = Pipeline([('scaler', StandardScaler())])
cat_transformer = Pipeline([('ohe', OneHotEncoder(handle_unknown='ignore', sparse=False))])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, numerical_cols),
        ('cat', cat_transformer, categorical_cols),
    ],
    remainder='drop'
)

# Train/test split (stratify not used since regression)
X = data[FEATURES].copy()
y = data[TARGET].copy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Example: fit preprocessor and transform once (the pipelines below will include it automatically)
# preprocessor.fit(X_train)


## Training multiple models in a loop and centralized evaluation

In [ ]:
def evaluate_regression(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {'rmse': rmse, 'mse': mse, 'mae': mae, 'mape': mape, 'r2': r2}

models = {
    'RandomForest': RandomForestRegressor(n_jobs=-1, random_state=42),
    'XGBoost': XGBRegressor(objective='reg:squarederror', n_jobs=-1, random_state=42, verbosity=0),
    'GradientBoosting': GradientBoostingRegressor(random_state=42)
}

results = {}
for name, estimator in models.items():
    print(f'Training {name}...')
    pipe = Pipeline([('preprocessor', preprocessor), ('model', estimator)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    metrics = evaluate_regression(y_test, y_pred)
    results[name] = metrics
    print(f'{name} metrics: ', metrics)

    # Save the pipeline
    model_path = os.path.join('models', f'superkart_pipeline_{name}.joblib')
    joblib.dump(pipe, model_path)
    print(f'Saved {name} pipeline to {model_path}
')

# Summary
print('
Training summary:')
for k,v in results.items():
    print(k, v)


## Save preprocessing object separately (optional)

In [ ]:
# Save preprocessor alone for quick inference pipeline assembly
joblib.dump(preprocessor, os.path.join('artifacts', 'preprocessor.joblib'))
print('Saved preprocessor to artifacts/preprocessor.joblib')


## Notes and next steps
- You can add GridSearchCV loops wrapping each pipeline if you want hyperparameter tuning.
- Consider using feature selection or target transformation if distributions are skewed.
- For very large OHE outputs, consider using frequency encoding or target encoding to reduce dimensionality.
